In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
bronze_path = "abfss://bronze@logisticdatalakestorage.dfs.core.windows.net/waqi/"
silver_path = "abfss://silver@logisticdatalakestorage.dfs.core.windows.net/waqi/"

In [0]:
df = spark.read.json(bronze_path)
df.limit(2).display()

data,status,year,month,day
"List(65, List(List(UK-Department-for-environment-food-and-rural-affairs.png, UK-AIR, air quality information resource - Defra, UK, http://uk-air.defra.gov.uk/), List(UK-London-Kings-College.png, London Air Quality Network - Environmental Research Group, King's College London, https://londonair.org.uk/), List(null, World Air Quality Index Project, https://waqi.info/)), List(List(51.5073509, -0.1277583), , London, https://aqicn.org/city/london), List(2026-05-27T17:35:56+09:00), pm25, List(List(List(List(14, 2026-05-25, 28, 2), List(16, 2026-05-26, 25, 6), List(14, 2026-05-27, 23, 7), List(15, 2026-05-28, 25, 6), List(12, 2026-05-29, 18, 7), List(10, 2026-05-30, 19, 2), List(12, 2026-05-31, 15, 6), List(12, 2026-06-01, 12, 9)), List(List(34, 2026-05-25, 46, 27), List(26, 2026-05-26, 36, 16), List(21, 2026-05-27, 33, 11), List(22, 2026-05-28, 36, 10), List(23, 2026-05-29, 40, 9), List(16, 2026-05-30, 31, 8), List(9, 2026-05-31, 32, 5), List(8, 2026-06-01, 9, 8)), List(List(40, 2026-05-25, 51, 27), List(33, 2026-05-26, 48, 24), List(38, 2026-05-27, 56, 18), List(29, 2026-05-28, 40, 17), List(28, 2026-05-29, 52, 12), List(27, 2026-05-30, 51, 12), List(16, 2026-05-31, 54, 9), List(16, 2026-06-01, 17, 16)), List(List(0, 2025-03-29, 0, 0), List(0, 2025-03-30, 4, 0), List(1, 2025-03-31, 5, 0), List(1, 2025-04-01, 5, 0), List(1, 2025-04-02, 4, 0), List(0, 2025-04-03, 3, 0), List(0, 2025-04-04, 3, 0), List(1, 2025-04-05, 4, 0)))), List(List(1.8), null, List(66.6), List(26.1), List(43.9), List(1027.2), List(41), List(65), null, List(1.1), List(20.1), List(5.4), null), 5724, List(2026-05-27T08:00:00+01:00, 2026-05-27 08:00:00, +01:00, 1779868800))",ok,2026,5,27
"List(28, List(List(Germany-Berlin.png, Berlin Air Quality - (Luftqualität in Berlin), http://www.stadtentwicklung.berlin.de/umwelt/luftqualitaet/), List(Europe-EEA.png, European Environment Agency, http://www.eea.europa.eu/themes/air/), List(Germany-UBA.png, Umweltbundesamt | Für Mensch und Umwelt, https://www.umweltbundesamt.de/), List(null, World Air Quality Index Project, https://waqi.info/)), List(List(52.5200066, 13.404954), , Berlin, Germany, https://aqicn.org/city/germany/berlin), List(2026-05-27T18:18:31+09:00), o3, List(List(List(List(12, 2026-05-25, 19, 4), List(15, 2026-05-26, 22, 4), List(15, 2026-05-27, 18, 12), List(13, 2026-05-28, 18, 9), List(14, 2026-05-29, 23, 3), List(15, 2026-05-30, 19, 12), List(13, 2026-05-31, 18, 5), List(10, 2026-06-01, 10, 6)), List(List(7, 2026-05-25, 8, 4), List(22, 2026-05-26, 39, 9), List(10, 2026-05-27, 35, 4), List(6, 2026-05-28, 7, 4), List(9, 2026-05-29, 10, 7), List(11, 2026-05-30, 18, 7), List(11, 2026-05-31, 14, 6), List(14, 2026-06-01, 14, 13)), List(List(13, 2026-05-25, 17, 7), List(26, 2026-05-26, 32, 18), List(15, 2026-05-27, 41, 7), List(11, 2026-05-28, 15, 8), List(18, 2026-05-29, 22, 13), List(24, 2026-05-30, 38, 13), List(14, 2026-05-31, 17, 12), List(18, 2026-06-01, 25, 18)), List(List(1, 2026-05-25, 8, 0), List(1, 2026-05-26, 7, 0), List(1, 2026-05-27, 7, 0), List(1, 2026-05-28, 7, 0), List(1, 2026-05-29, 7, 0), List(1, 2026-05-30, 6, 0), List(2, 2026-05-31, 6, 0)))), List(List(0.1), null, List(53.0), List(2.8), List(28.1), List(1022.7), List(11), List(21), null, null, List(20.5), List(9.0), null), 6132, List(2026-05-27T10:00:00+02:00, 2026-05-27 10:00:00, +02:00, 1779876000))",ok,2026,5,27


In [0]:
df_flat = df.select(
    col("data.city.name").alias("city"),
    col("data.city.geo")[0].alias("latitude"),
    col("data.city.geo")[1].alias("longitude"),
    col("data.dominentpol").alias("dominant_pollutant"),
    col("data.iaqi.pm25.v").alias("pm25"),
    col("data.iaqi.pm10.v").alias("pm10"),
    col("data.iaqi.o3.v").alias("o3"),
    col("data.iaqi.no2.v").alias("no2"),
    col("data.iaqi.co.v").alias("co"),
    col("data.iaqi.t.v").alias("temperature"),
    col("data.iaqi.h.v").alias("humidity"),
    col("data.time.iso").alias("measurement_time")
)

In [0]:
df_flat.limit(5).display()

city,latitude,longitude,dominant_pollutant,pm25,pm10,o3,no2,co,temperature,humidity,measurement_time
London,51.5073509,-0.1277583,pm25,65,41,43.9,26.1,1.8,20.1,66.6,2026-05-27T08:00:00+01:00
"Berlin, Germany",52.5200066,13.404954,o3,21,11,28.1,2.8,0.1,20.5,53.0,2026-05-27T10:00:00+02:00
Bangkok,13.7563309,100.5017651,pm25,39,22,22.5,4.7,0.1,37.0,52.0,2026-05-27T16:00:00+07:00
Paris,48.856614,2.3522219,pm25,72,25,34.4,45.8,0.1,21.1,61.0,2026-05-27T07:00:00+02:00
"Kurla, Mumbai, India",19.0863,72.8888,pm25,151,79,11.7,2.4,2.6,35.0,73.03,2026-05-27T12:00:00+05:30


In [0]:
df_flat = df_flat.withColumn(
    "measurement_time",
    to_timestamp(col("measurement_time"))
)
df_flat.limit(15).display()

city,latitude,longitude,dominant_pollutant,pm25,pm10,o3,no2,co,temperature,humidity,measurement_time
London,51.5073509,-0.1277583,pm25,65,41,43.9,26.1,1.8,20.1,66.6,2026-05-27T07:00:00.000Z
"Berlin, Germany",52.5200066,13.404954,o3,21,11,28.1,2.8,0.1,20.5,53.0,2026-05-27T08:00:00.000Z
Bangkok,13.7563309,100.5017651,pm25,39,22,22.5,4.7,0.1,37.0,52.0,2026-05-27T09:00:00.000Z
Paris,48.856614,2.3522219,pm25,72,25,34.4,45.8,0.1,21.1,61.0,2026-05-27T05:00:00.000Z
"Kurla, Mumbai, India",19.0863,72.8888,pm25,151,79,11.7,2.4,2.6,35.0,73.03,2026-05-27T06:30:00.000Z
"Central, Singapore",1.3666667,103.8,pm25,68,39,32.0,null,4.0,33.0,52.0,2026-05-27T09:00:00.000Z
New York,40.7127837,-74.0059413,pm25,30,null,null,null,null,20.0,84.0,2026-05-27T07:00:00.000Z
"Al Ain/Al Tawia, UAE",24.259283538347,55.704940681812,pm25,95,68,36.2,3.7,null,43.0,13.0,2026-05-27T08:00:00.000Z
Cook And Phillip Sydney East,-33.872468,151.213337,pm25,43,23,1.9,21.8,2.2,19.4,90.0,2026-05-27T07:00:00.000Z
"Parque D.Pedro II, São Paulo, Brazil",-23.544845659,-46.627675592,pm25,109,51,4.1,17.4,11.8,17.7,97.0,2026-05-27T09:00:00.000Z


In [0]:
silver_df = df_flat \
    .withColumn("year", year("measurement_time")) \
    .withColumn("month", month("measurement_time")) \
    .withColumn("day", dayofmonth("measurement_time"))

In [0]:
silver_df = silver_df \
    .withColumn('join_time', date_trunc('day', col('measurement_time'))) \
    .withColumn('ingestion_date', current_date()) \
    .dropDuplicates(['city', 'ingestion_date'])

In [0]:
silver_df.printSchema()

root
 |-- city: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- dominant_pollutant: string (nullable = true)
 |-- pm25: long (nullable = true)
 |-- pm10: long (nullable = true)
 |-- o3: double (nullable = true)
 |-- no2: double (nullable = true)
 |-- co: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- measurement_time: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- join_time: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = false)



In [0]:
silver_df = silver_df.drop('city')

In [0]:
BRONZE = 'abfss://bronze@logisticdatalakestorage.dfs.core.windows.net/'

df_cities_raw = spark.read.option('multiline', 'true').json(BRONZE + 'config/config.json')

df_city_ref = df_cities_raw \
    .select(explode(col('cities')).alias('c')) \
    .select(
        initcap(col('c.name')).alias('mapped_city'),
        upper(col('c.country')).alias('mapped_country'),
        col('c.lat').alias('c_lat'),
        col('c.lon').alias('c_lon')
    )

df_city_ref.show()

+-----------+--------------+--------+--------+
|mapped_city|mapped_country|   c_lat|   c_lon|
+-----------+--------------+--------+--------+
|     Mumbai|            IN|  19.076| 72.8777|
|     Berlin|            DE|   52.52|  13.405|
|     London|            GB| 51.5074| -0.1278|
|   New York|            US| 40.7128| -74.006|
|    Bangkok|            TH| 13.7563|100.5018|
|      Dubai|            AE| 25.2048| 55.2708|
|  Singapore|            SG|  1.3521|103.8198|
|     Sydney|            AU|-33.8688|151.2093|
|      Paris|            FR| 48.8566|  2.3522|
|  Sao Paulo|            BR|-23.5505|-46.6333|
+-----------+--------------+--------+--------+



In [0]:
silver_df = silver_df \
    .withColumn('latitude', col('latitude').cast('double')) \
    .withColumn('longitude', col('longitude').cast('double')) \
    .filter(col('latitude').isNotNull() & col('longitude').isNotNull())

In [0]:
window_city = Window.partitionBy('latitude', 'longitude', 'measurement_time').orderBy('_dist')

silver_df = silver_df \
    .crossJoin(broadcast(df_city_ref)) \
    .withColumn('_lat_diff', abs(col('latitude') - col('c_lat'))) \
    .withColumn('_lon_diff', abs(col('longitude') - col('c_lon'))) \
    .withColumn('_dist', col('_lat_diff') + col('_lon_diff')) \
    .filter(col('_dist') < 1.5) \
    .withColumn('_rn', row_number().over(window_city)) \
    .filter(col('_rn') == 1) \
    .drop('_lat_diff', '_lon_diff', '_dist', '_rn', 'c_lat', 'c_lon') \
    .withColumnRenamed('mapped_city', 'city') \
    .withColumnRenamed('mapped_country', 'country_code')

In [0]:
silver_df.display()

latitude,longitude,dominant_pollutant,pm25,pm10,o3,no2,co,temperature,humidity,measurement_time,year,month,day,join_time,ingestion_date,city,country_code
-33.872468,151.213337,pm25,43,23,1.9,21.8,2.2,19.4,90.0,2026-05-27T07:00:00.000Z,2026,5,27,2026-05-27T00:00:00.000Z,2026-05-27,Sydney,AU
-23.544845659,-46.627675592,pm25,109,51,4.1,17.4,11.8,17.7,97.0,2026-05-27T09:00:00.000Z,2026,5,27,2026-05-27T00:00:00.000Z,2026-05-27,Sao Paulo,BR
1.3666667,103.8,pm25,68,39,32.0,null,4.0,33.0,52.0,2026-05-27T09:00:00.000Z,2026,5,27,2026-05-27T00:00:00.000Z,2026-05-27,Singapore,SG
13.7563309,100.5017651,pm25,39,22,22.5,4.7,0.1,37.0,52.0,2026-05-27T09:00:00.000Z,2026,5,27,2026-05-27T00:00:00.000Z,2026-05-27,Bangkok,TH
19.0863,72.8888,pm25,151,79,11.7,2.4,2.6,35.0,73.03,2026-05-27T06:30:00.000Z,2026,5,27,2026-05-27T00:00:00.000Z,2026-05-27,Mumbai,IN
24.259283538347,55.704940681812,pm25,95,68,36.2,3.7,null,43.0,13.0,2026-05-27T08:00:00.000Z,2026,5,27,2026-05-27T00:00:00.000Z,2026-05-27,Dubai,AE
40.7127837,-74.0059413,pm25,30,null,null,null,null,20.0,84.0,2026-05-27T07:00:00.000Z,2026,5,27,2026-05-27T00:00:00.000Z,2026-05-27,New York,US
48.856614,2.3522219,pm25,72,25,34.4,45.8,0.1,21.1,61.0,2026-05-27T05:00:00.000Z,2026,5,27,2026-05-27T00:00:00.000Z,2026-05-27,Paris,FR
51.5073509,-0.1277583,pm25,65,41,43.9,26.1,1.8,20.1,66.6,2026-05-27T07:00:00.000Z,2026,5,27,2026-05-27T00:00:00.000Z,2026-05-27,London,GB
52.5200066,13.404954,o3,21,11,28.1,2.8,0.1,20.5,53.0,2026-05-27T08:00:00.000Z,2026,5,27,2026-05-27T00:00:00.000Z,2026-05-27,Berlin,DE


In [0]:
silver_df.write.format('delta').mode('overwrite').partitionBy("year", "month", "day").save(silver_path)